# JiT-S/2-Vim on CIFAR-10 — Stripped: Attention → BiMamba (Vim Algo. 1)

A direct ablation of the JiT-S/2 stripped notebook: **the multi-head self-attention mixer is replaced with the bidirectional Mamba mixer from Vision Mamba (Zhu et al., 2024)**, implemented faithfully to Algorithm 1 of the Vim paper.

Everything else in the architecture and recipe is unchanged from the stripped baseline:

- BottleneckPatchEmbed, RMSNorm, SwiGLU, adaLN-Zero, fixed 2D sin-cos posembed.
- No in-context class tokens (stripped).
- No CFG, no label dropout (stripped).
- x-prediction with v-loss, logit-normal `t`-sampler, Heun ODE sampler, dual EMA, `t_eps=0.05`, `noise_scale=1.0`.
- Same `Denoiser` wrapper, same checkpointing, same FID+IS+complexity+throughput evaluation.

**What changed in the model:**

- The `Attention` class is replaced with **`BiMambaV2`** — a faithful port of Vim Algorithm 1:
  - One shared `in_proj` produces `x` and `z`.
  - Forward and backward each have their own `Conv1d`, `x_proj` (Δ/B/C), `dt_proj`, `A_log`, `D`.
  - Both directions are gated by the **same** `SiLU(z)`, summed, then a **single** shared `out_proj` is applied.
- **RoPE removed** — Mamba has no queries/keys to rotate. Position information comes from the fixed sin-cos posembed (kept) and the sequential SSM scan order.
- **`expand=1`** ( not paper-faithful Vim Tiny/Small: `E = 2D`). This makes the mixer's inner state width `2 × 384 = 768`.

**Backend:** requires the official `mamba-ssm` CUDA kernel and `causal-conv1d`. **No pure-PyTorch fallback** — the notebook hard-fails if either package isn't importable. Both are installed in cell 1. This means the notebook needs a CUDA GPU (Kaggle T4/P100/Colab T4/local NVIDIA).

**Why the Vim *v1* implementation (which wraps two full `Mamba` modules and sums their outputs) is wrong:** see the comparison at the bottom of the BiMambaV2 cell. The Vim *v1* construction has two `in_proj`s, two `z` gates, two `out_proj`s — which is not Algorithm 1, and not any of the four bidirectional variants the paper ablates (Table 4).


## 0. Configuration rationale (read this first)

The architecture is the JiT scaffold (BottleneckPatchEmbed, RMSNorm, SwiGLU, adaLN-Zero, fixed sin-cos posembed) with the **token mixer swapped from attention to BiMamba**. Vim doesn't define an "S-class" model either, so the dimensions below are inherited from the JiT-S notebook with Vim's recommended SSM hyperparameters.

**Architecture (no in-context, no CFG, attention → BiMamba):**

| Knob | Value | Why |
|---|---|---|
| `input_size` | 32 | CIFAR-10 |
| `patch_size` | 2 | Same sequence length (16×16 = 256 tokens) as the attention baseline |
| `depth` | 12 | Same as the attention baseline / JiT-B |
| `hidden_size` | 384 | Same as the attention baseline |
| `bottleneck_dim` | 128 | Paper Table 9: 128 for B/L |
| `mlp_ratio` | 4.0 | SwiGLU internally rescales by ×2/3 |
| **Mixer** | **`BiMambaV2`** | Vim Algorithm 1: shared in_proj/out_proj, per-direction Conv1d+SSM, shared `z` gate, summed outputs |
| `d_state` (N) | 16 | Vim paper section 3.4 default |
| `d_conv` | 4 | Vim/Mamba default kernel size for the depthwise Conv1d |
| **`expand`** (E/D) | **2** | Vim paper section 3.4: `E = 2D` for both Tiny and Small. (Notebook 1 used `expand=1`; we use the paper-faithful value.) |
| `dt_rank` | `ceil(D/16) = 24` | Mamba default |
| `num_classes` | 10 | CIFAR-10 |
| in-context tokens | **removed** | Stripped |
| CFG / label dropout | **removed** | Stripped |
| RoPE | **removed** | Mamba has no Q/K; position info comes from sin-cos posembed + scan order |

**Flow / training (paper Table 9, unchanged from the stripped attention baseline):**

| Knob | Value | Notes |
|---|---|---|
| `P_mean`, `P_std` | -0.8, 0.8 | Logit-normal `t`-sampler |
| `t_eps` | 0.05 | Paper Table 9 |
| `noise_scale` | 1.0 | See attention baseline's rationale |
| `label_drop_prob` | 0.0 | CFG disabled |
| `sampler`, `steps` | Heun, 50 | Paper Table 9 |
| EMA decays | (0.9999, 0.9996) | `main_jit.py` defaults |
| optimizer | AdamW, β=(0.9, 0.95), wd=0 | Matches `main_jit.py` |
| lr rule | `actual_lr = blr × total_batch / 256` | `main_jit.py` |
| `PAPER_BLR` | 5e-5 | |
| effective `PEAK_LR` at batch 128 | 2.5e-5 | |
| warmup, schedule | 5 ep, constant | Paper Table 9 |
| epochs | 100 (default) | |
| batch size | 128 | |

**Note on parameter count.** Switching attention → BiMamba (`expand=2`) and dropping RoPE changes the per-block mixer dimensions. The total parameter count is reported by cell "Build model + Denoiser" below and again by cell 15.3 (complexity & throughput).

**Checkpointing:** see Section 11b — same as the attention baseline.


## 1. Env setup

Installs core PyTorch + **`mamba-ssm`** + **`causal-conv1d`**. The CUDA kernels are **required** — if either install fails, the notebook will hard-fail at the import step. There is no pure-PyTorch fallback by design (the Vim selective scan is slow enough in pure PyTorch that it's not worth running).

You need a CUDA GPU with a torch build matching the wheel ABI. On Kaggle (T4/P100) and Colab (T4) this normally just works.


In [ ]:


# === RUN ONCE, THEN RESTART RUNTIME ===

# 1) Pin torch to 2.5.1 (required for prebuilt mamba-ssm wheels)
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download prebuilt CUDA kernel wheels (no compilation needed)
import os
WHEELS = [
    ("https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/"
     "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"),
    ("https://github.com/state-spaces/mamba/releases/download/v2.2.4/"
     "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"),
]
for url in WHEELS:
    fname = url.split("/")[-1]
    if not os.path.exists(f"/kaggle/working/{fname}"):
        os.system(f"wget -q {url}")

!pip install -q /kaggle/working/causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/working/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl
!pip install -q pytorch-fid thop

# 3) Patch mamba-ssm utils/generation.py — it imports a transformers class that's been removed
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"✅ Patched {path}")

print("\n>>> RESTART RUNTIME NOW, then skip Cells 2-3 and run from Cell 4 onwards <<<")



## 2. Imports


In [ ]:
import math, copy, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image

import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"✅ torch {torch.__version__}  |  torchvision {torchvision.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─── Mamba CUDA kernels — REQUIRED. No PyTorch fallback. ─────────────────
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from causal_conv1d import causal_conv1d_fn

print("   mamba-ssm     : ✅")
print("   causal-conv1d : ✅")


## 3. Primitives: RMSNorm, 2D sin-cos posembed

RoPE is removed in the Vim variant — Mamba has no Q/K to rotate. Position information now comes from the fixed sin-cos posembed (kept) plus the sequential SSM scan order.


In [ ]:
# ─── RMSNorm ─────────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    """Root-mean-square LayerNorm, no bias."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).to(x.dtype)


# ─── Fixed 2D sin-cos positional embedding (DiT convention) ──────────────
def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.
    omega = 1. / 10000 ** omega
    pos = pos.reshape(-1)
    out = np.einsum('m,d->md', pos, omega)
    emb_sin = np.sin(out)
    emb_cos = np.cos(out)
    return np.concatenate([emb_sin, emb_cos], axis=1)


def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    return np.concatenate([emb_h, emb_w], axis=1)


def get_2d_sincos_pos_embed(embed_dim, grid_size):
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0).reshape([2, 1, grid_size, grid_size])
    return get_2d_sincos_pos_embed_from_grid(embed_dim, grid)


print("✅ RMSNorm + 2D sin-cos posembed defined (no RoPE — Vim has no Q/K)")


## 4. BottleneckPatchEmbed, BiMambaV2 (Vim Algorithm 1), SwiGLUFFN

The patch embed and SwiGLU FFN are unchanged. The attention mixer is replaced with `BiMambaV2`, a faithful port of Vim Algorithm 1.


In [ ]:
class BottleneckPatchEmbed(nn.Module):
    """Two-Conv2d bottleneck patch embed, matching model_jit.py exactly."""
    def __init__(self, img_size, patch_size, in_chans, bottleneck_dim, embed_dim, bias=True):
        super().__init__()
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.num_patches = (img_size // patch_size) ** 2
        self.proj1 = nn.Conv2d(in_chans, bottleneck_dim, kernel_size=patch_size,
                                stride=patch_size, bias=False)
        self.proj2 = nn.Conv2d(bottleneck_dim, embed_dim, kernel_size=1, stride=1, bias=bias)

    def forward(self, x):
        x = self.proj2(self.proj1(x))
        return x.flatten(2).transpose(1, 2)   # (B, N, embed_dim)


# ────────────────────────────────────────────────────────────────────────
# BiMambaV2 — Vim Algorithm 1, faithful
# ────────────────────────────────────────────────────────────────────────
#
# Algorithm 1 (Zhu et al. 2024, "Vision Mamba"):
#   T'_{l-1} = Norm(T_{l-1})
#   x = Linear_x(T'_{l-1})                                   ← ONE linear (shared)
#   z = Linear_z(T'_{l-1})                                   ← ONE linear (shared)
#   for o in {forward, backward}:
#       x'_o = SiLU(Conv1d_o(x))                             ← per-direction Conv1d
#       B_o, C_o, Δ_o ← Linear^o(x'_o)                       ← per-direction projections
#       y_o ← SSM(x'_o; A_o, B_o, C_o, Δ_o, D_o)             ← per-direction SSM
#   y'_fwd  = y_fwd  ⊙ SiLU(z)                               ← SHARED z gates both
#   y'_bwd  = y_bwd  ⊙ SiLU(z)
#   T_l = Linear_T(y'_fwd + y'_bwd) + T_{l-1}                ← SHARED out_proj, sum, residual
#
# CRITICAL DIFFERENCE vs. "wrap two full Mambas and sum":
#   ✅ ONE in_proj, ONE z, ONE out_proj  (shared across directions)
#   ❌ "two-Mamba" wrappers have two in_projs, two z's, two out_projs
#
# Per-block parameter cost at hidden=384, expand=2, d_state=16, d_conv=4, dt_rank=24:
#   in_proj  = 384 × 768       = 294,912        (no bias)
#   out_proj = 768 × 384       = 294,912        (no bias)
#   per direction (× 2):
#     Conv1d  = 768 × 4 + 768  =     3,840
#     x_proj  = 768 × (24+32)  =    43,008      (no bias)
#     dt_proj = 24 × 768 + 768 =    19,200
#     A_log   = 768 × 16       =    12,288
#     D       = 768            =       768
#   Total mixer params per block ≈ 1.04 M.


def _dt_init(dt_proj, d_inner, dt_init_floor=1e-4, dt_min=0.001, dt_max=0.1):
    """Vim/Mamba dt_proj init: bias is softplus^{-1} of uniform[dt_min, dt_max]."""
    dt_init_std = d_inner ** -0.5
    nn.init.uniform_(dt_proj.weight, -dt_init_std, dt_init_std)
    dt = torch.exp(
        torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
    ).clamp(min=dt_init_floor)
    inv_dt = dt + torch.log(-torch.expm1(-dt))   # softplus^{-1}(dt)
    with torch.no_grad():
        dt_proj.bias.copy_(inv_dt)
    dt_proj.bias._no_reinit = True


class _DirectionalSSM(nn.Module):
    """One scan direction: depthwise Conv1d → SiLU → selective_scan."""
    def __init__(self, d_inner, d_state, d_conv, dt_rank):
        super().__init__()
        self.d_inner, self.d_state, self.d_conv, self.dt_rank = d_inner, d_state, d_conv, dt_rank

        self.conv1d = nn.Conv1d(
            in_channels=d_inner, out_channels=d_inner,
            kernel_size=d_conv, padding=d_conv - 1,
            groups=d_inner, bias=True,
        )
        self.x_proj  = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        _dt_init(self.dt_proj, d_inner)

        # A_log: (E, N), A initialized to -[1..N] (S4 init)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True
        self.D = nn.Parameter(torch.ones(d_inner))
        self.D._no_weight_decay = True

    def forward(self, x_inner):
        """x_inner: (B, L, E) → (B, L, E)."""
        B_, L, E = x_inner.shape
        x_t = x_inner.transpose(1, 2).contiguous()        # (B, E, L)

        # Depthwise causal Conv1d with fused SiLU (causal-conv1d kernel)
        x_t = causal_conv1d_fn(
            x_t, self.conv1d.weight.squeeze(1), self.conv1d.bias,
            activation="silu",
        )
        x_after_conv = x_t.transpose(1, 2).contiguous()   # (B, L, E)

        # Project to (Δ, B_ssm, C_ssm)
        x_dbl = self.x_proj(x_after_conv)                                  # (B, L, dt_rank + 2N)
        dt, Bm, Cm = torch.split(x_dbl,
                                  [self.dt_rank, self.d_state, self.d_state],
                                  dim=-1)
        dt = self.dt_proj(dt)                                              # (B, L, E)

        A = -torch.exp(self.A_log.float())                                 # (E, N)

        # Selective-scan CUDA kernel.
        # signature: u(B,E,L), delta(B,E,L), A(E,N), B(B,N,L), C(B,N,L), D(E,)
        y = selective_scan_fn(
            x_after_conv.transpose(1, 2).contiguous(),
            dt.transpose(1, 2).contiguous(),
            A,
            Bm.transpose(1, 2).contiguous(),
            Cm.transpose(1, 2).contiguous(),
            self.D.float(),
            z=None,
            delta_bias=None,
            delta_softplus=True,
            return_last_state=False,
        )                                                                  # (B, E, L)

        return y.transpose(1, 2).contiguous()                              # (B, L, E)


class BiMambaV2(nn.Module):
    """Vim Algorithm 1 — bidirectional Mamba with full-sequence flip semantics.

    Shared in_proj produces a single (x, z). Two _DirectionalSSM stacks scan
    x_inner forward and backward independently. Both outputs are gated by the
    *same* SiLU(z), summed, and projected by a *shared* out_proj.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2,
                 dt_rank=None, proj_drop=0.0):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_state = d_state
        self.d_conv  = d_conv
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_model / 16)

        # Shared in_proj: D → 2*E (produces x_inner and z gate)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        # Per-direction SSM stacks
        self.ssm_fwd = _DirectionalSSM(self.d_inner, d_state, d_conv, self.dt_rank)
        self.ssm_bwd = _DirectionalSSM(self.d_inner, d_state, d_conv, self.dt_rank)
        # Shared out_proj
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        """x: (B, L, D) → (B, L, D)."""
        x = x.contiguous()
        xz = self.in_proj(x)                                # (B, L, 2E)
        x_inner, z = xz.chunk(2, dim=-1)                    # (B, L, E) each

        y_fwd = self.ssm_fwd(x_inner)                       # forward scan
        y_bwd = self.ssm_bwd(torch.flip(x_inner, dims=[1])).flip(dims=[1])

        z_act = F.silu(z)
        y = y_fwd * z_act + y_bwd * z_act                   # shared z gates both
        return self.proj_drop(self.out_proj(y))             # shared out_proj after sum


# ────────────────────────────────────────────────────────────────────────
# SwiGLU FFN — unchanged from the attention baseline
# ────────────────────────────────────────────────────────────────────────
class SwiGLUFFN(nn.Module):
    """JiT SwiGLU: hidden_dim is passed as mlp_ratio*hidden_size, internally rescaled by 2/3."""
    def __init__(self, dim, hidden_dim, drop=0.0, bias=True):
        super().__init__()
        hidden_dim = int(hidden_dim * 2 / 3)
        self.w12 = nn.Linear(dim, 2 * hidden_dim, bias=bias)
        self.w3  = nn.Linear(hidden_dim, dim, bias=bias)
        self.ffn_dropout = nn.Dropout(drop)

    def forward(self, x):
        x12 = self.w12(x)
        x1, x2 = x12.chunk(2, dim=-1)
        return self.w3(self.ffn_dropout(F.silu(x1) * x2))


print("✅ BottleneckPatchEmbed, BiMambaV2 (Vim Algo. 1), SwiGLUFFN defined")


## 5. TimestepEmbedder, LabelEmbedder

`LabelEmbedder` still has a `num_classes + 1` embedding table (slot for the null class), even though we don't use it. Keeping the table at +1 means a checkpoint from this stripped model is still shape-compatible with the full `denoiser.py` if you ever want to re-add CFG later.


In [ ]:
class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=t.device) / half
        )
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return emb

    def forward(self, t):
        return self.mlp(self.timestep_embedding(t, self.frequency_embedding_size))


class LabelEmbedder(nn.Module):
    """
    Reference LabelEmbedder: NO internal dropout. Label dropout happens
    externally in the flow wrapper (matching JiT's denoiser.py).
    """
    def __init__(self, num_classes, hidden_size):
        super().__init__()
        self.embedding_table = nn.Embedding(num_classes + 1, hidden_size)
        self.num_classes = num_classes

    def forward(self, labels):
        return self.embedding_table(labels)

print("✅ TimestepEmbedder, LabelEmbedder defined")


## 6. JiTBlock, FinalLayer

`JiTBlock` now wraps `BiMambaV2` as its mixer (replacing the attention call). No RoPE is passed because Mamba doesn't use Q/K. The adaLN-Zero conditioning, residual structure, and FinalLayer are unchanged.


In [ ]:
def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class JiTBlock(nn.Module):
    """JiT block with adaLN-Zero conditioning, BiMamba mixer (Vim Algo. 1), and SwiGLU FFN."""
    def __init__(self, hidden_size, num_heads=None, mlp_ratio=4.0,
                 d_state=16, d_conv=4, expand=2, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        # num_heads kept for signature parity with the attention baseline; unused.
        self.norm1 = RMSNorm(hidden_size, eps=1e-6)
        self.mixer = BiMambaV2(
            d_model=hidden_size,
            d_state=d_state, d_conv=d_conv, expand=expand,
            proj_drop=proj_drop,
        )
        self.norm2 = RMSNorm(hidden_size, eps=1e-6)
        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        self.mlp = SwiGLUFFN(hidden_size, mlp_hidden_dim, drop=proj_drop)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True),
        )

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN_modulation(c).chunk(6, dim=-1)
        x = x + gate_msa.unsqueeze(1) * self.mixer(
            modulate(self.norm1(x), shift_msa, scale_msa))
        x = x + gate_mlp.unsqueeze(1) * self.mlp(
            modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x


class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = RMSNorm(hidden_size)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True),
        )

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        return self.linear(x)


print("✅ JiTBlock (BiMamba mixer) + FinalLayer defined")


## 7. The JiT model (stripped — Vim mixer, no in-context tokens, no RoPE)

Same scaffold as the stripped attention baseline, with two changes:

- **No RoPE.** The `feat_rope` argument is gone from blocks because Mamba doesn't use Q/K.
- **Mamba knobs** (`d_state`, `d_conv`, `expand`) are exposed as constructor args.


In [ ]:
class JiT(nn.Module):
    """JiT with BiMamba mixer. No in-context tokens. Class conditioning via adaLN-Zero only."""
    def __init__(
        self,
        input_size=32,
        patch_size=2,
        in_channels=3,
        hidden_size=384,
        depth=12,
        num_heads=6,           # kept for signature parity with attention baseline; unused
        mlp_ratio=4.0,
        attn_drop=0.0,
        proj_drop=0.0,
        num_classes=10,
        bottleneck_dim=128,
        # Mamba knobs
        d_state=16,
        d_conv=4,
        expand=2,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = in_channels
        self.patch_size = patch_size
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_classes = num_classes

        # Embedders
        self.t_embedder = TimestepEmbedder(hidden_size)
        self.y_embedder = LabelEmbedder(num_classes, hidden_size)

        self.x_embedder = BottleneckPatchEmbed(
            input_size, patch_size, in_channels, bottleneck_dim, hidden_size, bias=True
        )

        # Fixed 2D sin-cos pos embed (no RoPE)
        num_patches = self.x_embedder.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, hidden_size),
                                       requires_grad=False)

        # Transformer blocks (with BiMamba mixer); middle-half dropout slot
        lo, hi = depth // 4, depth // 4 * 3
        self.blocks = nn.ModuleList([
            JiTBlock(
                hidden_size, num_heads=num_heads, mlp_ratio=mlp_ratio,
                d_state=d_state, d_conv=d_conv, expand=expand,
                attn_drop=attn_drop if (lo <= i < hi) else 0.0,
                proj_drop=proj_drop if (lo <= i < hi) else 0.0,
            )
            for i in range(depth)
        ])

        self.final_layer = FinalLayer(hidden_size, patch_size, self.out_channels)
        self.initialize_weights()

    def initialize_weights(self):
        def _basic_init(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        self.apply(_basic_init)

        # Fixed sin-cos pos embed
        pe = get_2d_sincos_pos_embed(
            self.pos_embed.shape[-1], int(self.x_embedder.num_patches ** 0.5)
        )
        self.pos_embed.data.copy_(torch.from_numpy(pe).float().unsqueeze(0))

        # Patch embed xavier init on flattened conv weights
        w1 = self.x_embedder.proj1.weight.data
        nn.init.xavier_uniform_(w1.view([w1.shape[0], -1]))
        w2 = self.x_embedder.proj2.weight.data
        nn.init.xavier_uniform_(w2.view([w2.shape[0], -1]))
        nn.init.constant_(self.x_embedder.proj2.bias, 0)

        # Embeddings
        nn.init.normal_(self.y_embedder.embedding_table.weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[2].weight, std=0.02)

        # adaLN-Zero
        for block in self.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].bias, 0)

        # Zero output
        nn.init.constant_(self.final_layer.linear.weight, 0)
        nn.init.constant_(self.final_layer.linear.bias, 0)

    def unpatchify(self, x):
        p = self.patch_size
        c = self.out_channels
        h = w = int(x.shape[1] ** 0.5)
        x = x.reshape(x.shape[0], h, w, p, p, c)
        x = torch.einsum("nhwpqc->nchpwq", x)
        return x.reshape(x.shape[0], c, h * p, h * p)

    def forward(self, x, t, y):
        """x: (B, C, H, W) | t: (B,) | y: (B,)  → (B, C, H, W)"""
        t_emb = self.t_embedder(t)
        y_emb = self.y_embedder(y)
        c = t_emb + y_emb

        x = self.x_embedder(x)
        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x, c)

        x = self.final_layer(x, c)
        return self.unpatchify(x)


print("✅ JiT (stripped, BiMamba mixer) defined")


## 8. `Denoiser` wrapper (stripped — no CFG, no label dropout)

What was removed from the faithful `Denoiser`:

- `cfg_scale`, `cfg_interval`, `label_drop_prob` constructor arguments — gone.
- `drop_labels()` method — gone.
- `forward()` no longer drops labels.
- `_forward_sample()` does a single conditional pass and returns `(x_pred − z) / (1 − t)`. No second pass with the null class. No interval mask. No `cfg_scale_interval` `where`.

`_euler_step`, `_heun_step`, `generate()`, `update_ema()`, and `swap_ema()` are unchanged. The "last step is always Euler" trick from the official sampler is preserved.

Net effect: training is plain class-conditional v-loss; sampling is plain Heun (or Euler) ODE integration with one network call per substep instead of two.


In [ ]:
class Denoiser(nn.Module):
    """
    JiT denoiser without CFG. Plain class-conditional training and sampling.
    Structurally still a port of LTH14/JiT denoiser.py, just with the CFG paths removed.
    """
    def __init__(
        self,
        net,                                   # the JiT module
        img_size: int,
        num_classes: int = 10,
        # flow hyperparameters (paper Table 9)
        P_mean: float = -0.8,
        P_std: float = 0.8,
        t_eps: float = 0.05,
        noise_scale: float = 1.0,
        # ema
        ema_decay1: float = 0.9999,
        ema_decay2: float = 0.9996,
        # sampling
        sampling_method: str = "heun",         # "heun" or "euler"
        num_sampling_steps: int = 50,
    ):
        super().__init__()
        self.net = net

        self.img_size = img_size
        self.num_classes = num_classes

        # flow
        self.P_mean = P_mean
        self.P_std = P_std
        self.t_eps = t_eps
        self.noise_scale = noise_scale

        # ema (populated lazily after model is on its final device)
        self.ema_decay1 = ema_decay1
        self.ema_decay2 = ema_decay2
        self.ema_params1 = None
        self.ema_params2 = None

        # sampling
        self.method = sampling_method
        self.steps = num_sampling_steps

    # ───────────────────────────── EMA ─────────────────────────────
    def init_ema(self):
        """Call once, after the model is on its final device."""
        with torch.no_grad():
            self.ema_params1 = [p.detach().clone() for p in self.parameters()]
            self.ema_params2 = [p.detach().clone() for p in self.parameters()]

    @torch.no_grad()
    def update_ema(self):
        source_params = list(self.parameters())
        for targ, src in zip(self.ema_params1, source_params):
            targ.detach().mul_(self.ema_decay1).add_(src, alpha=1 - self.ema_decay1)
        for targ, src in zip(self.ema_params2, source_params):
            targ.detach().mul_(self.ema_decay2).add_(src, alpha=1 - self.ema_decay2)

    def swap_ema(self, which: int = 1):
        ema = self.ema_params1 if which == 1 else self.ema_params2
        if ema is None:
            raise RuntimeError("EMA not initialized; call init_ema() first.")
        class _Swap:
            def __enter__(_self):
                _self.backup = [p.detach().clone() for p in self.parameters()]
                with torch.no_grad():
                    for p, e in zip(self.parameters(), ema):
                        p.copy_(e)
                return self
            def __exit__(_self, *a):
                with torch.no_grad():
                    for p, b in zip(self.parameters(), _self.backup):
                        p.copy_(b)
        return _Swap()

    # ───────────────────────── flow utilities ─────────────────────────
    def sample_t(self, n: int, device=None):
        z = torch.randn(n, device=device) * self.P_std + self.P_mean
        return torch.sigmoid(z)

    # ───────────────────────────── training ─────────────────────────────
    def forward(self, x, labels):
        """v-loss for one batch. No label dropout — CFG is disabled."""
        t = self.sample_t(x.size(0), device=x.device).view(-1, *([1] * (x.ndim - 1)))
        e = torch.randn_like(x) * self.noise_scale
        z = t * x + (1 - t) * e

        v = (x - z) / (1 - t).clamp_min(self.t_eps)
        x_pred = self.net(z, t.flatten(), labels)
        v_pred = (x_pred - z) / (1 - t).clamp_min(self.t_eps)

        loss = (v - v_pred) ** 2
        loss = loss.mean(dim=(1, 2, 3)).mean()
        return loss

    # ───────────────────────────── sampling ─────────────────────────────
    @torch.no_grad()
    def generate(self, labels, progress: bool = False):
        device = labels.device
        bsz = labels.size(0)
        z = self.noise_scale * torch.randn(bsz, 3, self.img_size, self.img_size, device=device)

        # (steps+1, bsz, 1, 1, 1) — same broadcast shape as denoiser.py
        timesteps = torch.linspace(0.0, 1.0, self.steps + 1, device=device)
        timesteps = timesteps.view(-1, *([1] * z.ndim)).expand(-1, bsz, -1, -1, -1)

        if self.method == "euler":
            stepper = self._euler_step
        elif self.method == "heun":
            stepper = self._heun_step
        else:
            raise NotImplementedError(self.method)

        iterator = range(self.steps - 1)
        if progress:
            iterator = tqdm(iterator, desc=f"{self.method.upper()}", leave=False)
        for i in iterator:
            t = timesteps[i]
            t_next = timesteps[i + 1]
            z = stepper(z, t, t_next, labels)

        # last step always Euler (matches denoiser.py)
        z = self._euler_step(z, timesteps[-2], timesteps[-1], labels)
        return z

    @torch.no_grad()
    def _forward_sample(self, z, t, labels):
        """Single conditional pass — no CFG, no unconditional branch."""
        x_pred = self.net(z, t.flatten(), labels)
        v_pred = (x_pred - z) / (1.0 - t).clamp_min(self.t_eps)
        return v_pred

    @torch.no_grad()
    def _euler_step(self, z, t, t_next, labels):
        v_pred = self._forward_sample(z, t, labels)
        return z + (t_next - t) * v_pred

    @torch.no_grad()
    def _heun_step(self, z, t, t_next, labels):
        v_pred_t = self._forward_sample(z, t, labels)
        z_next_euler = z + (t_next - t) * v_pred_t
        v_pred_t_next = self._forward_sample(z_next_euler, t_next, labels)
        v_pred = 0.5 * (v_pred_t + v_pred_t_next)
        return z + (t_next - t) * v_pred


print("✅ Denoiser (stripped — no CFG, no label dropout) defined")


## 9. Build model + Denoiser

`patch_size=2` → 256 tokens. Mamba mixer with `d_state=16`, `d_conv=4`, **`expand=2`** (paper-faithful Vim).


In [ ]:
IMG_SIZE     = 32
PATCH_SIZE   = 2          # → 16×16 = 256 tokens
NUM_CLASSES  = 10
HIDDEN       = 384
DEPTH        = 12
HEADS        = 6          # unused (kept for signature parity)
BOTTLENECK   = 128

# Mamba knobs (Vim paper section 3.4)
D_STATE = 16
D_CONV  = 4
EXPAND  = 1               # NOT E = 2D — paper-faithful Vim

net = JiT(
    input_size     = IMG_SIZE,
    patch_size     = PATCH_SIZE,
    in_channels    = 3,
    hidden_size    = HIDDEN,
    depth          = DEPTH,
    num_heads      = HEADS,
    mlp_ratio      = 4.0,
    attn_drop      = 0.0,
    proj_drop      = 0.0,
    num_classes    = NUM_CLASSES,
    bottleneck_dim = BOTTLENECK,
    d_state        = D_STATE,
    d_conv         = D_CONV,
    expand         = EXPAND,
).to(device)

denoiser = Denoiser(
    net                = net,
    img_size           = IMG_SIZE,
    num_classes        = NUM_CLASSES,
    P_mean             = -0.8,
    P_std              = 0.8,
    t_eps              = 0.05,
    noise_scale        = 1.0,
    ema_decay1         = 0.9999,
    ema_decay2         = 0.9996,
    sampling_method    = "heun",
    num_sampling_steps = 50,
).to(device)

denoiser.init_ema()

n_params = sum(p.numel() for p in net.parameters())
n_mixer  = sum(p.numel() for n, p in net.named_parameters() if 'mixer' in n)
print(f"JiT-S/2-Vim parameters: {n_params/1e6:.2f} M total")
print(f"  Mixer (BiMambaV2 × {DEPTH}): {n_mixer/1e6:.2f} M  "
      f"({n_mixer/DEPTH/1e6:.3f} M / block, expand={EXPAND})")
print(f"  Sequence length: {(IMG_SIZE // PATCH_SIZE) ** 2} tokens")
print(f"  Mamba backend  : CUDA (mamba-ssm + causal-conv1d, required)")


## 10. CIFAR-10 dataset

Normalize to [-1, 1] so the data has ~unit variance per pixel, matching `noise_scale=1.0`.


In [ ]:
CIFAR_PATH = "./data"

transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ToTensor(),                                  # [0, 1]
    T.Normalize(mean=[0.5, 0.5, 0.5],
                 std=[0.5, 0.5, 0.5]),             # → [-1, 1]
])

train_ds = torchvision.datasets.CIFAR10(
    root=CIFAR_PATH, train=True, download=True, transform=transform
)

BATCH_SIZE = 128
loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
print(f"CIFAR-10: {len(train_ds)} train images, batch size {BATCH_SIZE}")


## 11. Optimizer & LR schedule

Paper recipe at batch 1024: Adam β=(0.9, 0.95), wd=0, lr=2e-4, constant after 5-epoch warmup. We use a slightly larger effective lr at batch 128.


In [ ]:
NUM_EPOCHS    = 100
WARMUP_EPOCHS = 5

# Paper's exact LR rule (from main_jit.py):
#     actual_lr = blr * total_batch / 256,  with blr = 5e-5
# At total batch 1024 (8 GPUs × 128) the paper gets actual_lr = 2e-4 (Table 9).
# For our single-GPU setup at batch 128 the rule gives:
#     5e-5 * 128 / 256 = 2.5e-5
PAPER_BLR  = 5e-5
PEAK_LR    = PAPER_BLR * BATCH_SIZE / 256        # 2.5e-5 at batch 128

# Optimizer: the paper Table 9 says "Adam", but main_jit.py uses torch.optim.AdamW
# with weight_decay=0. With wd=0, AdamW and Adam are mathematically identical;
# we use AdamW to match the reference code byte-for-byte.
optimizer = torch.optim.AdamW(
    denoiser.parameters(),
    lr=PEAK_LR,
    betas=(0.9, 0.95),
    weight_decay=0.0,
)

steps_per_epoch = len(loader)
warmup_steps    = WARMUP_EPOCHS * steps_per_epoch

def lr_at_step(step: int) -> float:
    if step < warmup_steps:
        return PEAK_LR * (step + 1) / warmup_steps
    return PEAK_LR

scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
print(f"Optimizer ready. {steps_per_epoch} steps/epoch, {warmup_steps} warmup steps.")


## 11b. Checkpointing

Following `main_jit.py`'s convention:

- `checkpoint-last.pt` is overwritten every `SAVE_LAST_FREQ` epochs (default 5). This is the resume target if your runtime dies.
- `checkpoint-ep{N}.pt` is kept as a separate archived snapshot every `SAVE_ARCHIVE_FREQ` epochs (default 25), so you have history.
- `checkpoint-best.pt` is written whenever the epoch-average v-loss improves.

Each checkpoint includes: `epoch`, `global_step`, `model` (`denoiser.net` state_dict), `ema1` and `ema2` (the two EMA copies as state-dict-like ordered tensors), `optimizer`, and `losses` history.

The next cell defines helpers. The cell after that **optionally** resumes from a previous run — leave `RESUME_FROM = None` for a fresh start, or set it to a path like `"./checkpoints/checkpoint-last.pt"` to continue.


In [ ]:
import os
from collections import OrderedDict

CKPT_DIR          = "./checkpoints"
SAVE_LAST_FREQ    = 5     # overwrite checkpoint-last.pt every N epochs (for resume)
SAVE_ARCHIVE_FREQ = 25    # keep numbered checkpoint-ep{N}.pt every N epochs (for history)
os.makedirs(CKPT_DIR, exist_ok=True)

def _ema_to_state_dict(denoiser, which):
    """Convert ema_params{which} (a list of tensors aligned with parameters()) into
    an OrderedDict keyed by named_parameters() — so it's a regular state_dict shape."""
    ema = denoiser.ema_params1 if which == 1 else denoiser.ema_params2
    if ema is None:
        return None
    sd = OrderedDict()
    for (name, _), tensor in zip(denoiser.named_parameters(), ema):
        sd[name] = tensor.detach().cpu().clone()
    return sd

def _load_ema_from_state_dict(denoiser, state_dict, which):
    """Inverse of _ema_to_state_dict — write into denoiser.ema_params{which}."""
    if denoiser.ema_params1 is None or denoiser.ema_params2 is None:
        denoiser.init_ema()
    ema = denoiser.ema_params1 if which == 1 else denoiser.ema_params2
    name_to_idx = {name: i for i, (name, _) in enumerate(denoiser.named_parameters())}
    device = next(denoiser.parameters()).device
    with torch.no_grad():
        for name, tensor in state_dict.items():
            i = name_to_idx[name]
            ema[i].copy_(tensor.to(device))

def save_checkpoint(path, epoch, global_step, denoiser, optimizer, losses, best_loss):
    payload = {
        "epoch":        epoch,
        "global_step":  global_step,
        "model":        denoiser.net.state_dict(),
        "ema1":         _ema_to_state_dict(denoiser, 1),
        "ema2":         _ema_to_state_dict(denoiser, 2),
        "optimizer":    optimizer.state_dict(),
        "losses":       list(losses),
        "best_loss":    best_loss,
    }
    torch.save(payload, path)

def load_checkpoint(path, denoiser, optimizer=None, map_location=None):
    payload = torch.load(path, map_location=map_location or "cpu", weights_only=False)
    denoiser.net.load_state_dict(payload["model"])
    if payload.get("ema1") is not None:
        _load_ema_from_state_dict(denoiser, payload["ema1"], 1)
    if payload.get("ema2") is not None:
        _load_ema_from_state_dict(denoiser, payload["ema2"], 2)
    if optimizer is not None and "optimizer" in payload:
        optimizer.load_state_dict(payload["optimizer"])
    return payload

print(f"✅ Checkpoint helpers ready. Saving to {CKPT_DIR}/")
print(f"   - checkpoint-last.pt     (every {SAVE_LAST_FREQ} epochs, overwritten)")
print(f"   - checkpoint-ep{{N}}.pt    (every {SAVE_ARCHIVE_FREQ} epochs, kept)")
print(f"   - checkpoint-best.pt     (when epoch v-loss improves)")


### Optional: resume from a previous checkpoint

Set `RESUME_FROM` to a checkpoint path to continue training from where it left off. Leave as `None` for a fresh run. After loading you'll see the resumed `start_epoch`, `global_step`, and the loss-history length.


In [ ]:
RESUME_FROM = "/kaggle/input/datasets/rodamanthos2/last-jit-bimamba-cifar10/checkpoint-last-bimamba.pt"   # e.g. "./checkpoints/checkpoint-last.pt"

start_epoch  = 1
global_step  = 0
losses       = []
best_loss    = float("inf")

if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
    payload = load_checkpoint(RESUME_FROM, denoiser, optimizer=optimizer, map_location=device)
    start_epoch = payload["epoch"] + 1
    global_step = payload["global_step"]
    losses      = payload["losses"]
    best_loss   = payload.get("best_loss", float("inf"))
    print(f"✅ Resumed from {RESUME_FROM}")
    print(f"   start_epoch = {start_epoch}, global_step = {global_step}, "
          f"losses logged = {len(losses)}, best_loss = {best_loss:.4f}")
else:
    print("Starting from scratch (no resume).")


## 12. Training loop


In [ ]:
print(f"{'='*60}")
print(f"  Training JiT-S/2-Vim (stripped, BiMamba mixer) on CIFAR-10")
print(f"  {NUM_EPOCHS} epochs | batch {BATCH_SIZE} | lr {PEAK_LR}")
print(f"  No in-context tokens, no CFG, no label dropout — BiMamba mixer (Vim Algo. 1, expand=2)")
print(f"  Checkpoints → {CKPT_DIR}/  (last every {SAVE_LAST_FREQ} ep, "
      f"archive every {SAVE_ARCHIVE_FREQ} ep, best on improvement)")
print(f"  Resuming from epoch {start_epoch}" if start_epoch > 1 else "  Fresh run")
print(f"{'='*60}\n")

denoiser.train()
for epoch in range(start_epoch, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        for pg in optimizer.param_groups:
            pg["lr"] = lr_at_step(global_step)

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                loss = denoiser(images, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = denoiser(images, labels)
            loss.backward()
            optimizer.step()

        denoiser.update_ema()

        global_step += 1
        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    avg = epoch_loss / len(loader)
    losses.append(avg)
    msg = f"Epoch {epoch:3d}/{NUM_EPOCHS} | avg v-loss {avg:.4f}"

    # ── checkpointing ───────────────────────────────────────────────
    # 1) overwrite "last" every SAVE_LAST_FREQ epochs (and on the final epoch)
    if epoch % SAVE_LAST_FREQ == 0 or epoch == NUM_EPOCHS:
        save_checkpoint(f"{CKPT_DIR}/checkpoint-last.pt",
                        epoch, global_step, denoiser, optimizer, losses, best_loss)
        msg += "  [last]"

    # 2) keep a numbered archive every SAVE_ARCHIVE_FREQ epochs
    if epoch % SAVE_ARCHIVE_FREQ == 0:
        save_checkpoint(f"{CKPT_DIR}/checkpoint-ep{epoch:03d}.pt",
                        epoch, global_step, denoiser, optimizer, losses, best_loss)
        msg += f"  [archive ep{epoch:03d}]"

    # 3) "best" on epoch-average loss improvement
    if avg < best_loss:
        best_loss = avg
        save_checkpoint(f"{CKPT_DIR}/checkpoint-best.pt",
                        epoch, global_step, denoiser, optimizer, losses, best_loss)
        msg += f"  [best ↓ {best_loss:.4f}]"

    print(msg)

print("\n✅ Training complete.")


## 13. Loss curve


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("v-loss")
plt.title("JiT-S/2-Vim (stripped) CIFAR-10 — training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 14. Sample from EMA weights

No CFG means one network call per substep (Heun does two substeps per step, so 2 calls/step; the full-CFG version was 4 calls/step). Sampling is ~2× faster.


In [ ]:
denoiser.eval()

N_PER_CLASS = 8
classes = torch.arange(NUM_CLASSES, device=device).repeat_interleave(N_PER_CLASS)

with denoiser.swap_ema(1):
    samples = denoiser.generate(classes, progress=True)

samples = ((samples.clamp(-1, 1) + 1) / 2).cpu()

grid = make_grid(samples, nrow=N_PER_CLASS, padding=2)
plt.figure(figsize=(N_PER_CLASS, NUM_CLASSES))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis("off")
plt.title("JiT-S/2-Vim (stripped) on CIFAR-10 — EMA samples (rows = classes 0..9)", fontsize=12)
plt.tight_layout()
plt.show()


## 15. Evaluation: FID + Inception Score, complexity, and throughput

Three blocks below:

1. **Dump images** — write 10K real CIFAR-10 images and 10K generated images to disk (under EMA weights).
2. **FID + IS** — use `torch-fidelity` (same evaluator the official JiT repo uses); fall back to `pytorch-fid` + `torchmetrics` if `torch-fidelity` isn't available.
3. **Complexity & throughput** — params, MACs (via `thop`), GFLOPs, inference throughput at batch 128, and end-to-end sampling throughput at batch 16 with 50 Heun steps.

Set `N_SAMPLES = 50000` for the official-style 50K FID/IS (slower); `10000` is the usual quick check.


### 15.1 Dump 10K real + 10K generated images to disk


In [ ]:
import os
from torchvision.utils import save_image

REAL_DIR = "./fid_eval/real"
GEN_DIR  = "./fid_eval/generated"
os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(GEN_DIR,  exist_ok=True)

N_SAMPLES  = 10000        # set 50000 for official-style evaluation
BATCH_GEN  = 200

# ── 1) dump real CIFAR-10 images (from the same normalized dataset we trained on) ──
# Need raw [0,1] images on disk, so we de-normalize from [-1, 1].
print(f"Dumping {N_SAMPLES} real CIFAR-10 images to {REAL_DIR} ...")
written = 0
for img, _ in train_ds:
    if written >= N_SAMPLES:
        break
    save_image((img + 1) / 2, f"{REAL_DIR}/{written:05d}.png")
    written += 1
print(f"  ✅ {written} real images written")

# ── 2) generate samples under EMA weights ──
print(f"Generating {N_SAMPLES} samples (Heun {denoiser.steps} steps) to {GEN_DIR} ...")
denoiser.eval()
count = 0
with torch.no_grad(), denoiser.swap_ema(1):
    pbar = tqdm(total=N_SAMPLES, desc="Generating")
    while count < N_SAMPLES:
        bs = min(BATCH_GEN, N_SAMPLES - count)
        y  = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        imgs = denoiser.generate(y)
        imgs = ((imgs.clamp(-1, 1) + 1) / 2).cpu()
        for j in range(bs):
            save_image(imgs[j], f"{GEN_DIR}/{count + j:05d}.png")
        count += bs
        pbar.update(bs)
    pbar.close()
print(f"  ✅ {count} generated images written")


### 15.2 FID + IS

`torch-fidelity` is the evaluator the official JiT repo uses; it returns both FID and IS from a single call over the same image folders. If installation fails, the cell falls back to `pytorch-fid` for FID and `torchmetrics` for IS.


In [ ]:
# Install once (silently).
import subprocess, sys, importlib

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Try torch-fidelity first — gives FID and IS in one call.
try:
    importlib.import_module("torch_fidelity")
except ImportError:
    try:
        _install("torch-fidelity")
        importlib.invalidate_caches()
    except Exception as e:
        print(f"torch-fidelity install failed: {e}")

USE_TF = False
try:
    import torch_fidelity
    USE_TF = True
except ImportError:
    pass

if USE_TF:
    print("Using torch-fidelity (same tool as the official JiT repo)\n")
    metrics = torch_fidelity.calculate_metrics(
        input1=GEN_DIR,
        input2=REAL_DIR,
        cuda=(device == "cuda"),
        isc=True,                       # Inception Score on input1 (generated)
        fid=True,                       # Frechet Inception Distance
        kid=False,
        verbose=False,
        samples_find_deep=False,
    )
    fid_value = metrics["frechet_inception_distance"]
    is_mean   = metrics["inception_score_mean"]
    is_std    = metrics["inception_score_std"]
else:
    print("Falling back to pytorch-fid + torchmetrics for IS\n")
    try: importlib.import_module("pytorch_fid")
    except ImportError: _install("pytorch-fid")
    try: importlib.import_module("torchmetrics")
    except ImportError: _install("torchmetrics[image]")

    from pytorch_fid import fid_score
    fid_value = fid_score.calculate_fid_given_paths(
        [REAL_DIR, GEN_DIR],
        batch_size=256, device=device, dims=2048,
    )

    # IS via torchmetrics: stream the generated images through it.
    from torchmetrics.image.inception import InceptionScore
    from PIL import Image
    import glob
    inception = InceptionScore(normalize=True).to(device)
    paths = sorted(glob.glob(f"{GEN_DIR}/*.png"))
    bsz = 100
    for i in range(0, len(paths), bsz):
        batch = []
        for p in paths[i:i+bsz]:
            img = torch.from_numpy(np.array(Image.open(p).convert("RGB"))).permute(2, 0, 1).float() / 255.0
            batch.append(img)
        batch = torch.stack(batch).to(device)
        inception.update(batch)
    is_mean, is_std = inception.compute()
    is_mean, is_std = float(is_mean), float(is_std)

print(f"{'='*50}")
print(f"  FID-{N_SAMPLES//1000}K:   {fid_value:.2f}")
print(f"  IS-{N_SAMPLES//1000}K:    {is_mean:.2f} ± {is_std:.2f}")
print(f"{'='*50}")


### 15.3 Complexity & throughput

Params + MACs via `thop`, inference throughput at batch 128 (single forward pass), and end-to-end sampling throughput at batch 16 with the configured Heun steps.


In [ ]:
import time, subprocess, sys, importlib

try:
    importlib.import_module("thop")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"])
from thop import profile

print("--- Complexity & Throughput ---")
net.eval()
denoiser.eval()

# ── 1) MACs and params via thop ──
dummy_x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
dummy_t = torch.rand(1, device=device)
dummy_y = torch.randint(0, NUM_CLASSES, (1,), device=device)
macs, params = profile(net, inputs=(dummy_x, dummy_t, dummy_y), verbose=False)
gflops = (macs * 2) / 1e9   # 1 MAC = 2 FLOPs

print(f"  Model:           JiT-S/2 stripped (depth {DEPTH}, hidden {HIDDEN}, no in-context, no CFG)")
print(f"  Parameters:      {params/1e6:.2f} M")
print(f"  MACs  (fwd):     {macs/1e9:.4f} G")
print(f"  GFLOPs (fwd):    {gflops:.4f} G")
print("-" * 50)

# ── 2) Inference throughput (single forward pass at batch 128) ──
INF_BSZ   = 128
N_ITERS   = 50
x_batch   = torch.randn(INF_BSZ, 3, IMG_SIZE, IMG_SIZE, device=device)
t_batch   = torch.rand(INF_BSZ, device=device)
y_batch   = torch.randint(0, NUM_CLASSES, (INF_BSZ,), device=device)

with torch.no_grad():
    for _ in range(5):
        net(x_batch, t_batch, y_batch)        # warmup
    if device == "cuda": torch.cuda.synchronize()
    start = time.time()
    for _ in range(N_ITERS):
        _ = net(x_batch, t_batch, y_batch)
    if device == "cuda": torch.cuda.synchronize()
    inf_img_per_sec = (N_ITERS * INF_BSZ) / (time.time() - start)

# ── 3) End-to-end sampling throughput (Heun, configured steps, batch 16) ──
SAMP_BSZ = 16
with torch.no_grad():
    y_bench = torch.randint(0, NUM_CLASSES, (SAMP_BSZ,), device=device)
    # warmup
    _ = denoiser.generate(y_bench, progress=False)
    if device == "cuda": torch.cuda.synchronize()
    start = time.time()
    _ = denoiser.generate(y_bench, progress=False)
    if device == "cuda": torch.cuda.synchronize()
    samp_time = time.time() - start
    samp_img_per_sec = SAMP_BSZ / samp_time

print(f"  Inference throughput (batch={INF_BSZ}):  {inf_img_per_sec:.0f} img/s")
print(f"  Sampling throughput  (batch={SAMP_BSZ}, {denoiser.steps} {denoiser.method} steps): "
      f"{samp_img_per_sec:.2f} img/s  ({samp_time:.2f}s for {SAMP_BSZ} images)")
print("=" * 50)
